# Annotation test run (validation set)

Runs a small LLM annotation test against the human validation gold: five
validation dialogues, all prompt templates, one model. Logic lives in
`extension/scripts/` (`prompt_loader`, `extraction`, `scoring`). Every call
caches per dialogue to `extension/artifacts/extraction_cache/{split}/{model}/{prompt}/{id}.json`
and re-runs skip valid entries. No output-token cap is set; models use their
provider defaults, and realised cost is measured from OpenRouter usage
accounting per call.

**Before running:** `export OPENROUTER_API_KEY=...` in the launching terminal.


In [1]:
import os, sys
from pathlib import Path
_here = Path.cwd()
for _c in [_here, *_here.parents]:
    if (_c / "extension" / "artifacts").exists():
        os.chdir(_c); break
sys.path.insert(0, str(Path.cwd()))
print("cwd:", os.getcwd(), "| key set:", bool(os.environ.get("OPENROUTER_API_KEY")))


cwd: /Users/tandon.utsav2/Desktop/Experiment_1 | key set: True


In [2]:
from extension.scripts.load_annotation_data import load_dataset
from extension.scripts import prompt_loader, extraction, scoring

gold = load_dataset("extension/artifacts/annotation_dev_and_val_sets/validation_set.csv")
DIALOGUES = extraction.dialogues_from(gold, split="validation")
UNITS = scoring.units_by_dialogue(gold)
print(f"{len(DIALOGUES)} dialogues, {len(gold)} units")


78 dialogues, 544 units


In [5]:
TEST_MODEL = 'moonshotai/kimi-k3'         # any OpenRouter model slug
TEST_PROMPTS = ['P1_full_codebook']       # any subset of prompt stems, e.g.
                                          # ['P1_full_codebook','P2_condensed_codebook',
                                          #  'P3_minimal','P4_condensed_staged']
N_TEST_DIALOGUES = 78


In [6]:
TEST_DIALOGUES = DIALOGUES[:N_TEST_DIALOGUES]
for _p in TEST_PROMPTS:
    assert _p in prompt_loader.list_prompts(), f"unknown prompt {_p!r}; available: {prompt_loader.list_prompts()}"
print(f"test model: {TEST_MODEL} on dialogues "
      f"{[d['dialogue_id'] for d in TEST_DIALOGUES]} x prompts {TEST_PROMPTS}\n")

import json as _json
test_rows = []
for prompt in TEST_PROMPTS:
    for dlg in TEST_DIALOGUES:
        status = extraction.generate_annotation(prompt, TEST_MODEL, dlg)
        rec = _json.load(open(extraction.cache_path(TEST_MODEL, prompt,
                                                    dlg['dialogue_id'], dlg['split'])))
        print(f"  {prompt:22s} {dlg['dialogue_id']}: {status:8s} cost ${rec['cost_usd']:.4f}  "
              f"latency {rec['latency_s']:.1f}s")
    s = scoring.score_config(gold, TEST_MODEL, prompt,
                             [d['dialogue_id'] for d in TEST_DIALOGUES], n_boot=0)
    test_rows.append(s)
    print(f"  -> validity {s['valid_rate']:.0%} | macro-F1(P) {s['macro_f1_P']:.3f} "
          f"| alpha {s['alpha']:.3f} | ${s['usd_per_dialogue']:.4f}/dialogue\n")

import pandas as pd
summary = pd.DataFrame(test_rows)[['prompt','valid_rate','macro_f1_P','alpha',
                                   'usd_per_dialogue','latency_s']].round(3)
print(summary.to_string(index=False))


test model: moonshotai/kimi-k3 on dialogues [1, 21, 35, 79, 143, 178, 255, 270, 275, 289, 300, 306, 323, 344, 351, 356, 380, 434, 448, 494, 532, 554, 589, 617, 635, 656, 695, 736, 758, 779, 818, 822, 842, 862, 947, 958, 966, 980, 992, 1026, 1051, 1063, 1071, 1084, 1089, 1107, 1119, 1217, 1221, 1300, 1349, 1421, 1452, 1490, 1519, 1540, 1553, 1555, 1557, 1571, 1615, 1658, 1668, 1717, 1719, 1776, 1780, 1870, 1890, 1916, 1941, 2018, 2164, 2192, 2196, 2202, 2206, 2222] x prompts ['P1_full_codebook']

  P1_full_codebook       1: cached   cost $0.5037  latency 767.4s
  P1_full_codebook       21: cached   cost $0.4325  latency 93.3s
  P1_full_codebook       35: cached   cost $0.2101  latency 84.1s
  P1_full_codebook       79: cached   cost $0.3028  latency 90.0s
  P1_full_codebook       143: cached   cost $0.2166  latency 53.8s
  P1_full_codebook       178: cached   cost $0.2770  latency 414.7s
  P1_full_codebook       255: cached   cost $0.2294  latency 228.3s
  P1_full_codebook       270: ca

### Notes

Prompt files live in `extension/artifacts/annotation_prompts/`; dropping a
new `P*.md` there adds it to the run automatically. Every attempt is cached
with its raw output, full reasoning trace, validation errors, and usage,
so misreadings of the codebook can be diagnosed from the trace, and are
retried on the next execution. Scores on five dialogues are a plumbing check,
not a ranking.
